In [9]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# モデルとトークナイザーの読み込み
model = AutoModelForCausalLM.from_pretrained(
    # １回目 そのままのデータ
    # "./tunedModels/prototype/checkpoint-3145"
    
    # ２回目 空白やあああを削除したデータ
    "./tunedModels/prototypeV2/checkpoint-2955"
)
tokenizer = AutoTokenizer.from_pretrained(
    "cyberagent/open-calm-small"
)

# テンプレート（入力なし）
template = (
    "以下はユーザーからの質問です。入力はタスクで参照されている文章です。質問を適切に満たす応答を書きなさい。\n\n"
    "### 指示:\n{instruction}\n\n"
    "### 応答:\n{output}"
)

# 質問リスト（ここに自由に追加してOK！）
instructions = [
    "今どんな気持ちですか？",
    "最近、自分を褒めてあげたいことは？",
    "あなたにとって幸せとは何ですか？",
    "あなたにとって安心できる場所はどこですか？",
    "最近変わった考え方ってありますか？",
    "誰かに相談したいことってありますか？",
    "あなたの癒しって何ですか？",
    "心に残っている音や音楽はありますか？",
    "どんな瞬間にやりがいを感じますか？",
    "今日、心に残った一言は？"
]

# 各質問に対して応答を生成
for idx, instruction in enumerate(instructions, 1):
    d = {
        "instruction": instruction,
        "output": ""  # 出力は空欄にしておく
    }

    ptext = template.format_map(d)

    input_ids = tokenizer.encode(ptext, return_tensors="pt")
    start_pos = len(input_ids[0])

    with torch.no_grad():
        tokens = model.generate(
            input_ids,
            max_new_tokens=128,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    output = tokenizer.decode(tokens[0][start_pos:], skip_special_tokens=True)

    print(f"\n【Q{idx}】{instruction}")
    print(f"→ {output}")



【Q1】今どんな気持ちですか？
→ 今日は、とても幸せ

【Q2】最近、自分を褒めてあげたいことは？
→ いつもありがとう

【Q3】あなたにとって幸せとは何ですか？
→ 孤独

【Q4】あなたにとって安心できる場所はどこですか？
→ 海

【Q5】最近変わった考え方ってありますか？
→ 人前で話すこと

【Q6】誰かに相談したいことってありますか？
→ 私のこと好き?

【Q7】あなたの癒しって何ですか？
→ 幸せ

【Q8】心に残っている音や音楽はありますか？
→ おとうさん

【Q9】どんな瞬間にやりがいを感じますか？
→ 朝起きたら、すごく達成感を感じることができている

【Q10】今日、心に残った一言は？
→ 今、自分の心に残っている言葉は?
